<a href="https://colab.research.google.com/github/inotakt63/my-first-lp/blob/tools/textmakev1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 💎 超高精度AI文字起こし（Whisper Turbo安定版）
# @markdown ※このセルの左側にある「再生成マーク（▶）」を1回だけクリックして、1〜2分お待ちください。
# @markdown システムが起動すると、画面の一番下に【Running on public URL: 〜】という青いリンクが出現します。

import os
import sys

print("[1/3] AI文字起こしシステムをインストール中...（約1分かかります）")
# エラーの原因になりやすい不要な大量ログを非表示にしてインストール
!pip install -q git+https://github.com/openai/whisper.git
!pip install -q gradio

print("[2/3] 超高精度AIモデル（Whisper Turbo）を読み込み中...")
import whisper
import gradio as gr

# メモリクラッシュを完璧に防ぎ、large-v3と同等の精度を誇る最新の「turbo」モデルを固定
model = whisper.load_model("turbo")
print("[3/3] 🎉 システムの起動に成功しました！下の青いURLリンク（Running on public URL）をクリックしてください。")

# 秒数を「01:23」のようなタイムスタンプ形式に変換するヘルパー関数
def format_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    if hours > 0:
        return f"[{hours:02d}:{minutes:02d}:{secs:02d}] "
    else:
        return f"[{minutes:02d}:{secs:02d}] "

# 2. 文字起こしの処理を行う関数（タイムスタンプ機能対応版）
def transcribe_audio(audio_file, use_timestamp):
    if audio_file is None:
        return "⚠️ 音声ファイルが選択されていません。ファイルをドラッグ＆ドロップしてください。"
    try:
        # 文字起こし実行（日本語に完全特化）
        result = model.transcribe(audio_file, verbose=False, language="ja")

        # タイムスタンプを付ける場合
        if use_timestamp:
            output_lines = []
            for segment in result["segments"]:
                timestamp = format_time(segment["start"])
                output_lines.append(f"{timestamp}{segment['text']}")
            return "\n".join(output_lines)

        # タイムスタンプを付けない場合（通常モード）
        else:
            return result["text"]

    except Exception as e:
        return f"エラーが発生しました。ファイル形式を確認してください。\n詳細: {str(e)}"

# 3. グラフィカルで美しいWebアプリ画面（Gradio UI）の構築
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 💎 超高精度AI文字起こしスタジオ
        ### 〜 Zoom議事録・YouTube音声・スマホ録音を一撃でテキスト化 〜

        ---

        ### ⚠️【重要なお知らせ：初めて利用される方へ】
        本ツールは**「音声から言葉を1文字も漏らさずに、限界まで正確に抜き出すこと」**に特化しています。
        そのため、ここでの出力結果には**句読点（。や、）がなく、改行もされていませんが【完全に正常な動作】**です。

        この後に控える**「ステップ2：超速SNS量産メーカー（Gemini Canvas）」**に、ここで出たテキストをそのまま貼り付けるだけで、
        AIが自動で美しい句読点・改行を施し、一瞬で極上のSNSポストやブログ記事へと大改造します！安心してお進みください。

        ---

        ### 【操作手順】
        1. 左側のグレーのエリアに、文字起こししたい**音声または動画ファイル（mp3, m4a, wav, mp4対応）**をドラッグ＆ドロップします。
        2. 画面真ん中にある**「タイムスタンプを付ける」**のチェックボックスでお好みのモードを選択します。
        3. **「🔥 AI文字起こしをスタート」**という大きな青いボタンを1回クリックします。
        4. 右側のボックスにテキストが完成したら、ボックス右上の**「Copy」アイコン**を押し、次のツールへお持ちください！
        """
    )

    with gr.Row():
        with gr.Column():
            # ファイルアップロード領域
            input_audio = gr.Audio(
                label="🎵 ここにファイルを放り込んでください（mp3 / m4a / wav / mp4対応）",
                type="filepath"
            )

            # ★追加：タイムスタンプ選択オプション（デフォルトは初心者向けにチェックなし＝付けない）
            timestamp_check = gr.Checkbox(
                label="⏰ タイムスタンプを付ける（動画編集や議事録の振り返り用）",
                value=False
            )

            # 実行ボタン
            submit_btn = gr.Button("🔥 AI文字起こしをスタート", variant="primary")

        with gr.Column():
            # 結果出力領域
            output_text = gr.Textbox(
                label="📋 【文字起こし結果】（ここから自由にコピーできます）",
                placeholder="ここに自動でテキストが書き出されます。\n※タイムスタンプをONにすると、[01:23] のような時間付きで書き出されます！",
                show_copy_button=True, # ワンクリックでコピーできるボタンを標準装備
                lines=15
            )

    # ボタンをクリックした時の動作設定（引数に入力を2つ渡すように拡張）
    submit_btn.click(
        fn=transcribe_audio,
        inputs=[input_audio, timestamp_check],
        outputs=output_text
    )

# 4. 完全に独立した外部の専用WebサイトURL（.gradio.live）を自動発行する設定
demo.launch(share=True, inline=False)

[1/3] AI文字起こしシステムをインストール中...（約1分かかります）
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
[2/3] 超高精度AIモデル（Whisper Turbo）を読み込み中...
[3/3] 🎉 システムの起動に成功しました！下の青いURLリンク（Running on public URL）をクリックしてください。


/tmp/ipykernel_11382/3676974756.py:33: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://45d400356e8814b6d4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
